In [ ]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.path.abspath("./"))

print(f"Current work directory: {os.getcwd()}")

In [ ]:
from sklearn.pipeline import Pipeline
import pandas as pd
from catboost import Pool
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scanpy as sc
import joblib

In [ ]:
model = joblib.load('./Data/model/Compact_cat_after_tunning.pkl')

In [ ]:
df = pd.read_csv('./CSV/Bulk-seq/GSE252692_converted.csv', index_col='Gene_symbol')

In [ ]:
df_t = df.transpose().copy()

In [ ]:
df_t.columns.name = 'Samples'

In [ ]:
adata = sc.AnnData(df_t.astype(float))
adata.obs_names = df_t.index
adata.var_names = df_t.columns        

adata.var_names_make_unique()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
df_norm_log = pd.DataFrame(
    adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

In [ ]:
num_features = model.feature_names_

use_cols = [c for c in num_features if c in df_norm_log.columns]

df_sub = df_norm_log[use_cols].copy()

In [ ]:
y_pred = model.predict(df_sub)

In [ ]:
y_pred

In [ ]:
df_sub['Prediction'] = y_pred

In [ ]:
df_sub['Condition'] = df_sub.index.str.split('_').str[1:-1].str.join('_')

In [ ]:
condition_order = [
    'uninfected', 'OC43_3hpi', 'OC43_6hpi', 'OC43_9hpi',
    'OC43_12hpi', 'OC43_18hpi', 'OC43_24hpi', 'OC43_30hpi'
]
df_sub['Condition'] = pd.Categorical(df_sub['Condition'], categories=condition_order, ordered=True)

In [ ]:
baseline = df_sub[df_sub['Condition'] == 'uninfected']['Prediction'].mean()

In [ ]:
df_sub['Expression_Change'] = df_sub['Prediction'] - baseline

In [ ]:
conditions_order = ['uninfected', 'OC43_3hpi', 'OC43_6hpi', 'OC43_9hpi', 
                    'OC43_12hpi', 'OC43_18hpi', 'OC43_24hpi', 'OC43_30hpi']
stats = df_sub.groupby('Condition')['Prediction'].agg(['mean', 'std']).reindex(conditions_order) # Expression Change
means = stats['mean'].values
stds = stats['std'].values

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.utils import resample

# ---------------------------------------------------------
# 1. Data prepare
# ---------------------------------------------------------
means = np.array(means)
stds = np.array(stds)

x_labels_clean = ['Mock', '3hpi', '6hpi', '9hpi', '12hpi', '18hpi', '24hpi', '30hpi']
x_indices = np.arange(len(x_labels_clean))

# [B] Experiment data (Bresson et al.)
exp_means_aligned = np.array([np.nan, np.nan, np.nan, 5.07, 342.58, 1044.38, 1355.27, 3325.20])
exp_sds_aligned   = np.array([np.nan, np.nan, np.nan, 6.40, 173.35, 222.29, 132.04, 1091.65])

# ---------------------------------------------------------
# 2. Statistical correlation & Robustness
# ---------------------------------------------------------
valid_mask = ~np.isnan(exp_means_aligned)
model_valid = means[valid_mask]
exp_valid_log = np.log10(exp_means_aligned[valid_mask])

# Pearson
r_val, p_val = pearsonr(model_valid, exp_valid_log)

# Spearman
rho_val, p_val_spearman = spearmanr(model_valid, exp_valid_log)

# Bootstrap CI for Pearson r
np.random.seed(42)
n_bootstraps = 1000
bootstrapped_r = []
for _ in range(n_bootstraps):
    indices = resample(np.arange(len(model_valid)))
    # Avoid constant arrays which would result in NaN correlation
    if len(np.unique(model_valid[indices])) > 1 and len(np.unique(exp_valid_log[indices])) > 1:
        r, _ = pearsonr(model_valid[indices], exp_valid_log[indices])
        bootstrapped_r.append(r)

ci_lower = np.percentile(bootstrapped_r, 2.5)
ci_upper = np.percentile(bootstrapped_r, 97.5)

print(f"STATS_RESULT: Pearson r={r_val:.3f} (p={p_val:.3f}, CI: [{ci_lower:.3f}, {ci_upper:.3f}])")
print(f"STATS_RESULT: Spearman rho={rho_val:.3f} (p={p_val_spearman:.3f})")

# ---------------------------------------------------------
# 3. Graph
# ---------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(10, 7)) # Increased height

ax2 = ax1.twinx()
color_exp = 'gray'

bars = ax2.bar(x_indices, exp_means_aligned, yerr=exp_sds_aligned,
               color=color_exp, alpha=0.5, width=0.6, 
               capsize=4, error_kw={'ecolor': 'black', 'elinewidth': 1.5},
               label='Experimental Viral Load (Bresson et al.)')

ax2.set_ylabel('Virion Production (virions/cell/h)', color='black', fontsize=16, fontweight='bold')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='black', labelsize=15)

ax2.grid(False)

color_model = '#1f77b4'

ax1.plot(x_indices, means, color=color_model, marker='s', markersize=8,
         linewidth=3, label='Model Prediction (Log expression)')

ax1.fill_between(x_indices, means - stds, means + stds, 
                 color=color_model, alpha=0.2, label='Model Prediction SD')

ax1.set_xlabel('Time Post Infection', fontsize=16, fontweight='bold')
ax1.set_ylabel('Model predicted log expression', color=color_model, fontsize=16, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_model, labelsize=15)

ax1.set_xticks(x_indices)
ax1.set_xticklabels(x_labels_clean, fontsize=15) 

ax1.grid(True, axis='y', linestyle='--', alpha=0.5)

ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)

if p_val < 0.001:
    p_text = "$P < 0.001$"
else:
    p_text = f"$P = {p_val:.3f}$" 
    
if p_val_spearman < 0.001:
    p_text_spearman = "$P < 0.001$"
else:
    p_text_spearman = f"$P = {p_val_spearman:.3f}$" 

stats_text = (
    f"Pearson $r = {r_val:.2f}$ (95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]), {p_text}\n"
    f"Spearman $\\rho = {rho_val:.2f}$, {p_text_spearman}"
)

# Expand the y-axis upper limit to give space for the legend/box
y_min, y_max = ax1.get_ylim()
ax1.set_ylim(y_min, y_max + (y_max - y_min) * 0.25) # Give even more room (25% extra)

# Use va='top' to anchor from the top and adjust positions to the top left
ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, 
         fontsize=14, fontweight='bold', color='#333333', va='top', ha='left',
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="gray", alpha=0.9))

# --- Legend ---
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

# Place legend below the text box (adjusted to 0.81 so it has slightly more space below the text box)
ax1.legend([lines_1[0], bars], [labels_1[0], labels_2[0]], 
           loc='upper left', bbox_to_anchor=(0.02, 0.81), fontsize=14, frameon=True)

plt.tight_layout()
# FIXED FILE NAME TO 260602
plt.savefig('./Plot/OC43_expression_MRC-5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import session_info

session_info.show()